# Генерация музыки с BPE-токенизацией

Генеративные сети (RNN, LSTM, Transformer) с токенизацией по принципу BPE (Byte Pair Encoding).

MIDI переводится в поток базовых токенов (высота ноты `P<pitch>`, длительность `D<bucket>`, пауза `<REST>`). Затем обучается BPE: частые соседние пары токенов сливаются в составные токены — выученные музыкальные мотивы. Словарь строится по данным, а не вручную.


## Импорты

In [ ]:
import glob
import json
import time
from collections import Counter

import pretty_midi
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| устройство:", DEVICE)

## Параметры и базовая токенизация MIDI

In [ ]:
NUM_FILES = 15
MAX_BASE_TOKENS = 60_000
NUM_MERGES = 300
REST_GAP = 0.5
DUR_BUCKETS = [0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
SEQ_LENGTH = 64
BATCH_SIZE = 64
EPOCHS = 20
LR = 0.001


def dur_bucket(duration):
    for b in DUR_BUCKETS:
        if duration <= b:
            return b
    return DUR_BUCKETS[-1]


def midi_to_base_tokens(midi_path):
    try:
        midi = pretty_midi.PrettyMIDI(midi_path)
    except Exception:
        return []
    tokens = []
    for instrument in midi.instruments:
        if instrument.is_drum:
            continue
        notes = sorted(instrument.notes, key=lambda n: n.start)
        for i, note in enumerate(notes):
            tokens.append(f"P{note.pitch}")
            tokens.append(f"D{dur_bucket(note.end - note.start)}")
            if i + 1 < len(notes) and notes[i + 1].start - note.end > REST_GAP:
                tokens.append("<REST>")
    return tokens

## BPE-токенизатор

Сегменты ограничиваются `<REST>` — слияния не пересекают паузы (аналог границ слов в текстовом BPE).

In [ ]:
class BPETokenizer:
    SPECIALS = ["<PAD>", "<UNK>", "<START>", "<END>"]

    def __init__(self, num_merges=NUM_MERGES):
        self.num_merges = num_merges
        self.merges = []
        self.token_to_idx = {}
        self.idx_to_token = {}

    def split_segments(self, tokens):
        segments, cur = [], []
        for t in tokens:
            if t == "<REST>":
                if cur:
                    segments.append(cur)
                    cur = []
                segments.append(["<REST>"])
            else:
                cur.append(t)
        if cur:
            segments.append(cur)
        return segments

    @staticmethod
    def count_pairs(segments):
        pairs = Counter()
        for seg in segments:
            for a, b in zip(seg, seg[1:]):
                pairs[(a, b)] += 1
        return pairs

    @staticmethod
    def merge_segment(seg, pair, new_token):
        a, b = pair
        out, i = [], 0
        while i < len(seg):
            if i + 1 < len(seg) and seg[i] == a and seg[i + 1] == b:
                out.append(new_token)
                i += 2
            else:
                out.append(seg[i])
                i += 1
        return out

    def fit(self, tokens):
        segments = self.split_segments(tokens)
        base_vocab = sorted({t for seg in segments for t in seg})
        print(f"Базовый алфавит: {len(base_vocab)}, сегментов: {len(segments)}")

        for step in range(self.num_merges):
            pairs = self.count_pairs(segments)
            if not pairs:
                break
            best, freq = pairs.most_common(1)[0]
            if freq < 2:
                break
            new_token = best[0] + "|" + best[1]
            self.merges.append(best)
            segments = [self.merge_segment(s, best, new_token) for s in segments]
            if (step + 1) % 50 == 0:
                print(f"Слияние {step + 1}: {best} (freq={freq})")

        vocab = self.SPECIALS + base_vocab + [a + "|" + b for a, b in self.merges]
        seen, ordered = set(), []
        for t in vocab:
            if t not in seen:
                seen.add(t)
                ordered.append(t)
        self.token_to_idx = {t: i for i, t in enumerate(ordered)}
        self.idx_to_token = {i: t for t, i in self.token_to_idx.items()}
        print(f"Словарь BPE: {len(self.token_to_idx)} токенов")

    def apply_merges(self, tokens):
        segments = self.split_segments(tokens)
        for pair in self.merges:
            new_token = pair[0] + "|" + pair[1]
            segments = [self.merge_segment(s, pair, new_token) for s in segments]
        return [t for seg in segments for t in seg]

    def encode(self, tokens):
        unk = self.token_to_idx["<UNK>"]
        return [self.token_to_idx.get(t, unk) for t in self.apply_merges(tokens)]

    def decode(self, indices):
        return [self.idx_to_token.get(i, "<UNK>") for i in indices]

    @property
    def vocab_size(self):
        return len(self.token_to_idx)

## Датасет и модели

In [ ]:
class MusicalDataset(Dataset):
    def __init__(self, sequences, seq_length=SEQ_LENGTH):
        self.sequences = sequences
        self.seq_length = seq_length

    def __len__(self):
        return max(0, len(self.sequences) - self.seq_length)

    def __getitem__(self, idx):
        x = self.sequences[idx:idx + self.seq_length]
        y = self.sequences[idx + 1:idx + self.seq_length + 1]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


class MusicRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        out, _ = self.rnn(self.embedding(x))
        return self.fc(out)


class MusicLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        out, _ = self.lstm(self.embedding(x))
        return self.fc(out)


class MusicTransformer(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, num_heads=4, num_layers=3, max_seq_len=1000):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.pos_encoding = nn.Parameter(torch.zeros(1, max_seq_len, embedding_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim, nhead=num_heads,
            dim_feedforward=embedding_dim * 4, dropout=0.1, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):
        seq_len = x.shape[1]
        emb = self.embedding(x) + self.pos_encoding[:, :seq_len, :]
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(x.device)
        return self.fc(self.transformer(emb, mask=mask))

## Обучение и генерация

In [ ]:
def train_model(model, loader, vocab_size, epochs=EPOCHS, lr=LR):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    losses = []
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out.reshape(-1, vocab_size), y.reshape(-1))
            loss.backward()
            optimizer.step()
            total += loss.item()
        avg = total / max(1, len(loader))
        losses.append(avg)
        if (epoch + 1) % 5 == 0:
            print(f"  epoch {epoch + 1}/{epochs}, loss {avg:.4f}")
    return losses


def generate_music(model, tokenizer, length=200, temperature=0.8):
    model.eval()
    forbidden = {tokenizer.token_to_idx[t] for t in BPETokenizer.SPECIALS}
    start = next(i for i in tokenizer.idx_to_token if i not in forbidden)
    current, generated = [start], []
    with torch.no_grad():
        for _ in range(length):
            inp = torch.tensor(current[-SEQ_LENGTH:]).unsqueeze(0).to(DEVICE)
            logits = model(inp)[0, -1, :] / temperature
            for idx in forbidden:
                logits[idx] = -float("inf")
            probs = torch.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1).item()
            generated.append(nxt)
            current.append(nxt)
    return tokenizer.decode(generated)

## Загрузка данных

In [ ]:
midi_files = sorted(glob.glob("data/maestro-v3.0.0/**/*.mid", recursive=True) +
                    glob.glob("data/maestro-v3.0.0/**/*.midi", recursive=True))
selected = midi_files[:NUM_FILES]
print(f"Найдено MIDI: {len(midi_files)}, используем: {len(selected)}")

## Базовая токенизация корпуса

In [ ]:
base_tokens = []
for path in selected:
    base_tokens.extend(midi_to_base_tokens(path))
    if len(base_tokens) >= MAX_BASE_TOKENS:
        break
base_tokens = base_tokens[:MAX_BASE_TOKENS]
print(f"Базовых токенов: {len(base_tokens)}, уникальных: {len(set(base_tokens))}")
print("Пример:", base_tokens[:12])

## Обучение BPE и кодирование

In [ ]:
t0 = time.time()
tokenizer = BPETokenizer(NUM_MERGES)
tokenizer.fit(base_tokens)
print(f"BPE обучен за {time.time() - t0:.1f} c")

sequences = tokenizer.encode(base_tokens)
compression = len(base_tokens) / len(sequences)
print(f"Длина последовательности: {len(sequences)}, сжатие x{compression:.2f}")

with open("bpe_vocab.json", "w", encoding="utf-8") as f:
    json.dump(tokenizer.token_to_idx, f, ensure_ascii=False, indent=2)
with open("bpe_merges.json", "w", encoding="utf-8") as f:
    json.dump(tokenizer.merges, f, ensure_ascii=False)

print("Примеры составных токенов:", list(tokenizer.token_to_idx)[-5:])

## DataLoader

In [ ]:
dataset = MusicalDataset(sequences, SEQ_LENGTH)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
vocab_size = tokenizer.vocab_size
print(f"Батчей: {len(loader)}, размер словаря: {vocab_size}")

## Обучение трёх моделей

In [ ]:
model_defs = {
    "RNN": MusicRNN(vocab_size),
    "LSTM": MusicLSTM(vocab_size),
    "Transformer": MusicTransformer(vocab_size),
}
history, trained = {}, {}
for name, model in model_defs.items():
    print(f"\n{name}")
    t0 = time.time()
    history[name] = train_model(model, loader, vocab_size)
    trained[name] = model
    torch.save(model.state_dict(), f"model_{name.lower()}_bpe.pth")
    print(f"{name}: {time.time() - t0:.1f} c, loss {history[name][-1]:.4f}")

## График сравнения

In [ ]:
plt.figure(figsize=(12, 6))
colors = {"RNN": "blue", "LSTM": "green", "Transformer": "red"}
for name, losses in history.items():
    plt.plot(losses, label=name, color=colors[name], linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Сравнение моделей (BPE-токенизация музыки)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig("training_comparison_bpe.png", dpi=150, bbox_inches="tight")
plt.show()

for name, losses in history.items():
    print(f"{name:12} loss {losses[-1]:.4f}")

## Генерация музыки

In [ ]:
for name, model in trained.items():
    print(f"\n{name}")
    for temp in [0.6, 0.8, 1.0]:
        gen = generate_music(model, tokenizer, length=150, temperature=temp)
        with open(f"generated_{name.lower()}_bpe_temp{temp}.txt", "w", encoding="utf-8") as f:
            f.write(" ".join(gen))
        print(f"  temp={temp}: {' '.join(gen[:8])} ...")

## Результаты

15 файлов MAESTRO, 60 000 базовых токенов, 300 BPE-слияний, словарь 391, сжатие x2.07, 20 эпох, CPU.

| Модель | Финальная Loss | Время |
|--------|----------------|-------|
| RNN         | 0.3492 | ~295 c |
| LSTM        | **0.1629** | ~840 c |
| Transformer | 0.3308 | ~931 c |

Лучшая модель по loss — LSTM. При таком объёме данных и числе эпох LSTM лучше всех улавливает локальные зависимости; трансформеру для преимущества нужно больше данных и эпох.

![Сравнение](training_comparison_bpe.png)
